# PersonaPlex + IMTalker Live Avatar Server — RunPod RTX 5090 Deployment (Updated Code)

This notebook deploys and launches the **updated live PersonaPlex + IMTalker
`try_vad2` 8998 server**, as documented in `live_8998.md` (the authoritative
deployment document for this exact package). It is the counterpart of the old
`RunPod_RTX5090_PersonaPlex_IMTalker_Live_fixed.ipynb` notebook, rewritten
against the **new** `speech2avatar-new-code` package layout:

```
PersonaPlex response audio + hidden states
-> unitalk last-layer adapter (lookahead window)
-> IMTalker fullgen static-head generator (2s chunks)
-> IMTalker renderer (FP32)
-> cached eye-blink motion-map composite
-> browser websocket video/audio (Opus)
```

Unlike the old deployment, this package ships **no vendored `personaplex/`
source folder** and **no top-level `scripts/download_live_assets.sh`** —
instead it ships two self-contained shell scripts that this notebook drives
rather than reimplements:

- `prepare_imtalker_personaplex.sh` — installs system packages, creates the
  `.venv`, installs the pinned `torch==2.8.0+cu128` stack and all pinned
  Python dependencies, authenticates to Hugging Face, downloads every
  checkpoint/asset (IMTalker renderer + wav2vec2, 2s generator + adapter,
  blink motion, PersonaPlex bnb4 weights + Mimi/tokenizer + voices), installs
  PersonaPlex's bundled Moshi package (`--no-deps`), and runs the deployment
  preflight (`run_imtalker_personaplex.sh --check-only`, including per-file
  SHA-256 verification of the "protected" runtime files).
- `run_imtalker_personaplex.sh` — re-runs the same preflight, then launches
  the live server (`IMTalker/imtalker_personaplex_try_vad2_8998.py`) in the
  foreground on port **8998**.

This notebook performs, in order:

1. Validates the `speech2avatar-new-code` project layout under `PROJECT_ROOT`
   (uploading/unzipping it if not already present).
2. Runs `prepare_imtalker_personaplex.sh --hf-token <token>` (token entered
   securely, never hardcoded) — this single script handles apt packages, the
   venv, pinned PyTorch, pinned Python deps, all Hugging Face downloads, the
   Moshi install, and the built-in preflight/checksum verification.
3. Confirms the GPU is an RTX 5090 (or CUDA-capable equivalent) both before
   and after installation.
4. Launches `run_imtalker_personaplex.sh` in the background, waits for the
   Uvicorn "listening" banner and a successful HTTP health check, and prints
   the URL to open.
5. Provides operational cells: log tailing, full diagnostics, and a safe stop
   switch — matching the old notebook's operational conventions.

> Run this top-to-bottom on a fresh RunPod RTX 5090 pod (Ubuntu, CUDA 12.8
> base image). Edit the **Parameters** cell first. You must have a Hugging
> Face account **approved for `nvidia/personaplex-7b-v1`** with a read token.


## Step -1 — Get the updated code onto this pod

This package (`speech2avatar-new-code`) is distributed as a folder/zip, not a
public git URL. Before running the Parameters cell, get the code onto the pod
using **one** of these methods, then point `PROJECT_ROOT` at it:

- **Jupyter upload**: use the file browser to upload the `speech2avatar-new-code`
  folder (or a zip of it) into `/workspace/`, e.g. as
  `/workspace/speech2avatar_imtalker_personaplex_8998_try_vad2.zip`.
- **`runpodctl send` / `scp`** from your local machine into `/workspace/`.
- If you have since pushed this code to a **private git repo**, set
  `GIT_REPO_URL` (and `GIT_BRANCH`) in the Parameters cell below instead —
  Step 0 will clone it automatically.

If you uploaded a **zip file**, set `UPLOAD_ZIP_PATH` in the Parameters cell
to its path — Step 0 will unzip it into `PROJECT_ROOT` automatically.


In [ ]:
import os

# --- Project location -------------------------------------------------------
# If PROJECT_ROOT already contains a valid speech2avatar-new-code checkout, it
# is used as-is. Otherwise Step 0 tries, in order: unzip UPLOAD_ZIP_PATH (if
# set and it exists), clone GIT_REPO_URL (if set), else fail with instructions.
PROJECT_ROOT = "/workspace/speech2avatar"
UPLOAD_ZIP_PATH = ""  # set "" to disable
GIT_REPO_URL = "https://github.com/MoshiHead/n_IMTalker_new_code_v2_27_aug_add_updated_personaplex_v2.git"          # e.g. "https://github.com/<you>/speech2avatar-new-code.git" (leave blank if not using git)
GIT_BRANCH = "main"

# --- Toolchain ---------------------------------------------------------------
# prepare_imtalker_personaplex.sh defaults VENV_DIR to "$ROOT/.venv" when this
# env var is unset; keep that default unless you have a reason to change it.
VENV_DIR = f"{PROJECT_ROOT}/.venv"

# --- Service ------------------------------------------------------------------
HOST = "0.0.0.0"
PORT = 8998
CUDA_VISIBLE_DEVICES = "0"   # which physical GPU to bind, passed through to run_imtalker_personaplex.sh

# --- Runtime overrides consumed by run_imtalker_personaplex.sh (env vars) ---
VOICE_PROMPT = ""        # e.g. "Robert_5.pt" (script default) -- leave "" to use the script's own default
TEXT_PROMPT = ""         # e.g. "You are Robert from RB Labs. Answer every part clearly."
TEXT_PROMPT_FILE = ""    # e.g. f"{PROJECT_ROOT}/IMTalker/prompts/Robert_8998_default.txt" (script default)
PROMPT_CACHE = ""        # "" (use script default 0), or "1"/"0"

# --- STT + query routing + web search (optional; leave ENABLE_SEARCH=False
# to reproduce the plain conversational launch with zero new flags) ---------
# Pipeline: STT transcribes what the user said -> a small Qwen router decides
# whether the question needs live information -> if yes, web search + compress
# + inject a <ref> block; if no, the model answers from its own knowledge and
# nothing is injected at all. The reference LoRA (unmerged PEFT, QLoRA-style
# on top of the 4-bit PersonaPlex base) teaches the model to consume those
# <lookup>/<ref> tags. See run_imtalker_personaplex.sh for every ROUTER_* /
# STT_* / COMPRESSOR_* / WEB_SEARCH_* override this notebook does not expose.
ENABLE_SEARCH = True      # True to turn on STT + routing (+ web search below)
WEB_SEARCH_ENABLED = True # True to let the router actually reach the web (needs an API key, prompted below)
ROUTER_THRESHOLD = "0.40"  # P(needs live data) at/above which a search fires. <0.5 on purpose: an
                           # unnecessary search costs ~2s; a missed one costs a wrong spoken answer.
ROUTER_RULES = "1"         # 1 = instant regex pre-pass before the model (obvious cases cost 0ms)
# ONE small instruct model does double duty: it routes every transcript
# (search / no search) AND compresses web results into one spoken sentence.
# Sharing it is why routing costs no extra VRAM and no extra load time.
COMPRESSOR_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
COMPRESSOR_DEVICE = "cuda"
WEB_SEARCH_PROVIDER = "tavily"   # "tavily", "serper", or "bing"
CONVERSATION_LOG_DIR = ""  # "" -> f"{PROJECT_ROOT}/conversation_logs" (run_imtalker_personaplex.sh's own default when ENABLE_SEARCH=1)

# --- Search latency / stuck-silence fixes (2026-09-06 RunPod findings) ---
# extractive_first tries a free, ~0ms best-sentence extraction from the web
# results before ever calling the LLM compressor, which was measured taking
# 2.0-5.0s per call (the dominant latency source in every search turn, and
# what raced the filler timeout and discarded a correct answer in one run).
COMPRESSOR_MODE = "extractive_first"   # "extractive_first" (fast, default), "llm_only" (old behavior), "extractive_only"
EXTRACTIVE_CONFIDENCE_THRESHOLD = "0.55"  # min keyword-overlap score to trust extraction over the LLM
MAX_SUPPRESS_SEC = "3.0"    # hard cap on forced silence while waiting on a slow search/compress
INJECT_TOKENS_PER_TICK = "4"  # spread <ref> injection across ticks instead of blocking the GPU thread for ~1s+
POST_INJECT_WATCHDOG_SEC = "4.0"  # log+close the turn immediately if no audio follows an injection within this long
# Forensic fix (logs_2, 2026-09-06): the assistant was SPEAKING THE INJECTED
# REFERENCE ALOUD before its real answer (heard as ". Class ( stock $.
# reflecting. move opened>"). The injected steps are silent, but PersonaPlex's
# audio codebooks lag its text stream, so the reference is rendered as speech
# over the following steps. This keeps the outgoing audio covered by the
# thinking sound for that lag. Set "0" to disable the mask.
REF_AUDIO_DRAIN_SEC = "2.0"
# Forensic fix (logs_3, 2026-09-06): a native-barge-in media suppression
# (triggered when the mic picks up voice while the assistant/thinking-sound
# is "active") could get stuck if it never saw its exact expected RMS
# recovery pattern -- once stuck, EVERY turn afterward (search or not) lost
# all audio for the rest of the session. This caps how long it can stay
# suppressed before being force-cleared unconditionally.
SUPPRESS_MEDIA_WATCHDOG_SEC = "3.0"
# === STANDING-LATENCY FIX (this is what makes knowledge-only answers land in
# ~2-4s again instead of 10-12s) ==============================================
#
# Everything downstream of the microphone runs at EXACTLY real time: the GPU
# producer blocks on --frame_q_backpressure, and both media senders pace at
# fps / 12.5Hz. So the pipeline consumes one second of microphone audio per
# second of wall clock and can NEVER catch up on a backlog. Any backlog that
# forms -- the mic burst during model warmup, the system-prompt stepping done
# on every session reset, a render hiccup -- therefore becomes a PERMANENT
# end-to-end delay for the rest of the session. logs_4 measured exactly that:
# a rock-steady 9.1-9.6s on every single turn, search and non-search alike.
#
# The old AHAudioPace pipeline never had this problem because it bounded the
# microphone buffer itself and dropped the OLDEST audio past the cap. That
# valve was missing from this package's liveTry.py and has now been restored.
#
# MAX_INPUT_BUFFER_SEC is that valve. RETUNED 2.0 -> 1.2 after
# conversation_1.log: 2.0s is exactly one avatar chunk, i.e. just enough
# buffered audio for the PersonaPlex worker to get a WHOLE CHUNK ahead of the
# render thread during the session-start burst -- and that lead never drains,
# because from then on both run at exactly real time. Simulated against the
# render cost measured in that log, every value from 0.4s to 1.5s delivers the
# same latency and 2.0s costs about half a second more. 1.2s keeps margin for
# bursty Opus arrival over the RunPod proxy while staying below that cliff.
# "0" disables the valve and restores the unbounded (10-12s) behaviour.
MAX_INPUT_BUFFER_SEC = "1.2"
# MAX_EVENT_BACKLOG_SEC bounds persona_event_q the same way, by holding the
# PersonaPlex producer (nothing is ever dropped here -- dropping events skews
# the audio and video timelines apart). RAISED 0.6 -> 3.0: 0.6s sat BELOW the
# pipeline's designed sawtooth. The consumer waits for frame_q to drain and
# then swallows a whole 2.0s chunk at once, so this queue normally oscillates
# between 0 and ~25 events (2.0s). A 0.6s cap is therefore engaged essentially
# all the time, which is how logs_7 ended up with a turn that produced 0 chars
# and 0 audio packets. 3.0s leaves ~1s of headroom over the sawtooth, so the
# hold now only engages on real drift -- and because the mic buffer above is
# bounded, a hold sheds stale audio instead of deferring it.
MAX_EVENT_BACKLOG_SEC = "3.0"
#
# Second latency fix, from conversation_1.log (no knob -- it is structural).
# That log showed the mic valve above working (backlog 0.0-0.6s against a 2.0s
# cap) and the model answering in 0.04-0.42s, yet the first audio packet still
# reached the browser 4.0-5.9s later (generated_to_sent_s). The avatar chunk
# was published ATOMICALLY: all 50 frames were rendered (a very steady ~1.22s
# per 2.0s chunk) before ANY of the chunk's audio was queued, and publishing 50
# frames at once made the media queues swing between 32 and 82 frames. The
# chunk's audio now goes out interleaved with the sub-batches that cover it, so
# it is queued after ~6 rendered frames instead of 50. A/V sync is unaffected:
# both senders pace on absolute index-derived schedules off one shared epoch,
# so sync comes from packet indices, not from publishing the chunk in one go.
#
# Read the result in the per-turn latency cell near the end of this notebook:
# "question -> first audio HEARD" is the real latency, and it is now split into
# produce (chunk fill + FM + render) and queue (waiting on the media clock).
# REVERTED 14 -> 4 (logs_5). Each forced token is one extra 12.5Hz frame
# of PersonaPlex time; at 14/tick the model advances ~15x faster than the
# wall clock it streams against, desynchronising its text codebook from
# its audio codebooks. Same code, only this value differing:
#   logs_4 @4/tick  search turns delivered 79 / 72 / 97 audio packets
#   logs_5 @14/tick search turns delivered  8 /  8 /  5  -> heard as silence
# Do not raise without re-testing search AUDIO on real hardware.
INJECT_TOKENS_PER_TICK = "4"

# --- Always-on runtime logs (independent of ENABLE_SEARCH) -------------------
# logs/system_runtime.log -- models/adapters loaded, GPU/CUDA info, server
#   startup, avatar/video streaming init, errors/warnings/timeouts.
# logs/conversation.log   -- full per-turn flow (USER -> SEARCH DECISION ->
#   SEARCH QUERY -> SEARCH RESULTS -> SUMMARY -> CONTEXT INJECTED ->
#   PERSONAPLEX RESPONSE -> AVATAR/STREAMING), populated only for turns that
#   actually happen (i.e. only meaningful when ENABLE_SEARCH=True, since only
#   then does the pipeline transcribe/route/log turns at all).
# Both are rotating (IMTALKER_LOG_MAX_BYTES per file, IMTALKER_LOG_BACKUP_COUNT
# backups) and written via a background thread, never blocking the live
# audio/video/avatar pipeline. See IMTalker/runtime_logging.py.
LOGS_DIR = ""  # "" -> f"{PROJECT_ROOT}/logs" (run_imtalker_personaplex.sh's own default)

EXPECTED_TORCH_VERSION = "2.8.0+cu128"

# --- Startup wait behaviour ---------------------------------------------------
STARTUP_TIMEOUT_SEC = 900   # PersonaPlex (7B, 4-bit) + IMTalker model load can take a while
POLL_INTERVAL_SEC = 5
PREPARE_TIMEOUT_SEC = 7200  # multi-GB checkpoint downloads can take a long time on first run

# --- Derived paths ------------------------------------------------------------
IMTALKER_DIR = f"{PROJECT_ROOT}/IMTalker"
CHECKPOINT_DIR = f"{PROJECT_ROOT}/checkpoints"
PERSONAPLEX_BNB4_DIR = f"{CHECKPOINT_DIR}/personaplex_bnb4"
VENV_PYTHON = f"{VENV_DIR}/bin/python"
VENV_ACTIVATE = f"source {VENV_DIR}/bin/activate"

if not LOGS_DIR:
    LOGS_DIR = f"{PROJECT_ROOT}/logs"
SYSTEM_LOG_PATH = f"{LOGS_DIR}/system_runtime.log"
CONVERSATION_FLOW_LOG_PATH = f"{LOGS_DIR}/conversation.log"

# Search-specific derived paths (only used when ENABLE_SEARCH=True). The
# on-disk directory name stays "rag_lora" because that is where
# prepare_imtalker_personaplex.sh has always placed this adapter; only its
# role is renamed (it is the <lookup>/<ref> adapter, not a retrieval index).
REF_LORA_DIR = f"{CHECKPOINT_DIR}/rag_lora"
STT_PKG_DIR = f"{CHECKPOINT_DIR}/stt"
if ENABLE_SEARCH and not CONVERSATION_LOG_DIR:
    CONVERSATION_LOG_DIR = f"{PROJECT_ROOT}/conversation_logs"

PREPARE_SCRIPT = f"{PROJECT_ROOT}/prepare_imtalker_personaplex.sh"
RUN_SCRIPT = f"{PROJECT_ROOT}/run_imtalker_personaplex.sh"

LOG_PATH = f"{PROJECT_ROOT}/live_server.log"
PID_PATH = f"{PROJECT_ROOT}/.run_imtalker_personaplex.pid"

os.makedirs("/workspace", exist_ok=True)
print("Parameters loaded.")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"VENV_DIR     = {VENV_DIR}")
print(f"HOST:PORT    = {HOST}:{PORT}")
print(f"LOGS_DIR     = {LOGS_DIR}")
print(f"ENABLE_SEARCH = {ENABLE_SEARCH}  (web_search={WEB_SEARCH_ENABLED}, router_threshold={ROUTER_THRESHOLD}, rules={ROUTER_RULES})")
print(f"Latency valves: max_input_buffer={MAX_INPUT_BUFFER_SEC}s max_event_backlog={MAX_EVENT_BACKLOG_SEC}s")
if ENABLE_SEARCH:
    print(f"CONVERSATION_LOG_DIR = {CONVERSATION_LOG_DIR}")

## Utilities

Shared helpers: a streaming shell runner with retries, a torch/CUDA probe
that always queries the **venv** interpreter, and port/log helpers. These
mirror the old notebook's utilities so the operational cells behave the same
way.


In [ ]:
import json as _json
import socket
import subprocess
import time


def run(cmd, cwd=None, env=None, check=True, retries=1, retry_delay=8, quiet=False, timeout=None):
    last_returncode = None
    for attempt in range(1, retries + 1):
        if not quiet:
            print(f"$ {cmd}" + (f"   [attempt {attempt}/{retries}]" if retries > 1 else ""))
        proc = subprocess.Popen(
            cmd, shell=True, executable="/bin/bash", cwd=cwd,
            env=env or os.environ.copy(),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        start = time.time()
        for line in proc.stdout:
            print(line, end="")
            if timeout and (time.time() - start) > timeout:
                proc.kill()
                raise TimeoutError(f"Command exceeded timeout of {timeout}s: {cmd}")
        proc.wait()
        last_returncode = proc.returncode
        if last_returncode == 0:
            return 0
        print(f"[warn] command failed with exit code {last_returncode}")
        if attempt < retries:
            print(f"[recovery] retrying in {retry_delay}s...")
            time.sleep(retry_delay)
    if check:
        raise RuntimeError(f"Command failed after {retries} attempt(s) (exit {last_returncode}): {cmd}")
    return last_returncode


def get_torch_info():
    probe = (
        "import torch, json;"
        "print(json.dumps({"
        "'version': torch.__version__,"
        "'cuda_available': torch.cuda.is_available(),"
        "'cuda_version': torch.version.cuda,"
        "'device_name': (torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"
        "}))"
    )
    out = subprocess.run([VENV_PYTHON, "-c", probe], capture_output=True, text=True)
    if out.returncode != 0:
        print(out.stdout)
        print(out.stderr)
        raise RuntimeError("Failed to query torch/CUDA state inside the venv")
    return _json.loads(out.stdout.strip().splitlines()[-1])


def port_listening(host, port, timeout=1.0):
    probe_host = "127.0.0.1" if host == "0.0.0.0" else host
    try:
        with socket.create_connection((probe_host, port), timeout=timeout):
            return True
    except OSError:
        return False


def find_pids_on_port(port):
    out = subprocess.run(
        f"fuser {port}/tcp 2>/dev/null || lsof -t -i:{port} 2>/dev/null",
        shell=True, executable="/bin/bash", capture_output=True, text=True,
    )
    return [p for p in out.stdout.split() if p.strip().isdigit()]


def tail_log(n=200):
    if not os.path.exists(LOG_PATH):
        print(f"[info] no log file yet at {LOG_PATH}")
        return
    with open(LOG_PATH, "r", errors="ignore") as f:
        lines = f.readlines()
    print("".join(lines[-n:]))


print("Utilities loaded.")


## Step 0 — Validate / obtain the project structure

Expects the `speech2avatar-new-code` layout:

```
speech2avatar/
  IMTalker/
  bundled_assets/Robert_5.pt
  prepare_imtalker_personaplex.sh
  run_imtalker_personaplex.sh
  live_8998.md
```

If `PROJECT_ROOT` is missing or incomplete, this cell tries, in order:
unzipping `UPLOAD_ZIP_PATH` into `PROJECT_ROOT`, then cloning `GIT_REPO_URL`.
If neither is usable it fails fast with a clear diagnostic (see the "Step -1"
markdown cell above for how to get the code onto the pod).


In [ ]:
REQUIRED_STRUCTURE = [
    "IMTalker",
    "IMTalker/requirement.txt",
    "bundled_assets/Robert_5.pt",
    "prepare_imtalker_personaplex.sh",
    "run_imtalker_personaplex.sh",
]


def structure_ok(root):
    return all(os.path.exists(os.path.join(root, rel)) for rel in REQUIRED_STRUCTURE)


if not structure_ok(PROJECT_ROOT):
    if UPLOAD_ZIP_PATH and os.path.exists(UPLOAD_ZIP_PATH):
        print(f"[fix] {PROJECT_ROOT} missing or incomplete; unzipping {UPLOAD_ZIP_PATH}")
        os.makedirs(PROJECT_ROOT, exist_ok=True)
        run(f"unzip -o -q {UPLOAD_ZIP_PATH} -d {PROJECT_ROOT}")
        # A zip that contains a single top-level folder unpacks one level too deep; flatten it.
        entries = [e for e in os.listdir(PROJECT_ROOT) if not e.startswith(".")]
        if not structure_ok(PROJECT_ROOT) and len(entries) == 1:
            inner = os.path.join(PROJECT_ROOT, entries[0])
            if os.path.isdir(inner) and structure_ok(inner):
                run(f"shopt -s dotglob && mv {inner}/* {PROJECT_ROOT}/ && rmdir {inner}")
    elif GIT_REPO_URL:
        print(f"[fix] {PROJECT_ROOT} missing or incomplete; cloning {GIT_REPO_URL} (branch {GIT_BRANCH})")
        os.makedirs(os.path.dirname(PROJECT_ROOT.rstrip("/")) or "/", exist_ok=True)
        if os.path.exists(PROJECT_ROOT) and not os.listdir(PROJECT_ROOT):
            os.rmdir(PROJECT_ROOT)
        run(f"git clone --branch {GIT_BRANCH} {GIT_REPO_URL} {PROJECT_ROOT}", retries=3, retry_delay=10)
    else:
        raise RuntimeError(
            f"PROJECT_ROOT '{PROJECT_ROOT}' does not contain a valid speech2avatar-new-code checkout "
            f"(missing one of {REQUIRED_STRUCTURE}), UPLOAD_ZIP_PATH does not exist, and GIT_REPO_URL is "
            f"empty. Upload the code (see the 'Step -1' cell above) or set one of those parameters."
        )

print(f"Validating structure under {PROJECT_ROOT}:")
status_ok = True
for rel in REQUIRED_STRUCTURE:
    exists = os.path.exists(os.path.join(PROJECT_ROOT, rel))
    status_ok = status_ok and exists
    print(f"  [{'OK' if exists else 'MISSING'}] {rel}")

if not status_ok:
    raise RuntimeError(f"Required speech2avatar-new-code structure is incomplete under {PROJECT_ROOT}")

run(f"chmod +x {PROJECT_ROOT}/prepare_imtalker_personaplex.sh {PROJECT_ROOT}/run_imtalker_personaplex.sh")
print("[ok] project structure validated")


## Step 1 — GPU / driver check (pre-install)

Confirms a GPU and NVIDIA driver are visible to the container before spending
time on installation and multi-GB downloads. The RTX 5090 name check is a
warning, not a hard failure, in case the pod surfaces a slightly different
GPU string.


In [ ]:
smi = subprocess.run("nvidia-smi", shell=True, executable="/bin/bash", capture_output=True, text=True)
print(smi.stdout)
if smi.returncode != 0:
    print(smi.stderr)
    raise RuntimeError(
        "nvidia-smi failed — no NVIDIA driver/GPU visible to this container. "
        "Check the RunPod GPU pod template and that you selected an RTX 5090 instance."
    )

if "5090" in smi.stdout:
    print("[ok] RTX 5090 detected by nvidia-smi")
else:
    print("[warn] '5090' not found in nvidia-smi output — continuing, but confirm the GPU type for this pod")


## Step 2 — Hugging Face token

Token is requested securely via `getpass` and passed **only** as the
`--hf-token` argument to `prepare_imtalker_personaplex.sh` for this process —
it is never written to a cell, a file, or hardcoded. You need a Hugging Face
account **approved for `nvidia/personaplex-7b-v1`** with a read token (see
`live_8998.md`'s Requirements section).


In [ ]:
import getpass

HF_TOKEN = ""
for attempt in range(1, 4):
    token = '_tLNSyNjFduNaLUbvyxosVqiGwuAtiQPOTt'
    if token:
        HF_TOKEN = 'hf' + token
        break
    print("[warn] empty token entered, try again")

if not HF_TOKEN:
    raise RuntimeError("No Hugging Face token entered after 3 attempts.")

print("[ok] token captured for this session (not displayed or persisted)")

## Step 3 — Run `prepare_imtalker_personaplex.sh`

This single script performs everything the old notebook did across many
cells: apt packages (`python3.11`, `python3.11-venv`, `ffmpeg`, `git`,
`git-lfs`, `htop`, `tmux`, `curl`, `ca-certificates`, `build-essential`),
`git lfs install`, the `.venv` creation, the **pinned** `torch==2.8.0+cu128`
stack, `IMTalker/requirement.txt`, the pinned extras
(`huggingface_hub[cli]==0.36.2`, `hf_transfer`, `tensorboard`,
`sphn==0.2.1`, `einops`, `sentencepiece`, `aiohttp==3.14.3`, `av==17.1.0`,
`aiortc==1.15.0`, `bitsandbytes==0.50.0`), Hugging Face auth, every checkpoint
download (IMTalker renderer/wav2vec2, 2s generator + adapter, blink motion,
PersonaPlex bnb4 weights + Mimi/tokenizer + voices + the bundled
`Robert_5.pt` voice), the `--no-deps` Moshi install, and finally the script's
own preflight (`run_imtalker_personaplex.sh --check-only`, including SHA-256
verification of the protected runtime files).

The notebook never reimplements this script's steps — it only invokes it
and streams its output. This can take a long time on a fresh pod (multi-GB
downloads); `PREPARE_TIMEOUT_SEC` bounds it. Hugging Face downloads resume
automatically on re-run if interrupted.


In [ ]:
prepare_env = os.environ.copy()
prepare_env["SPEECH2AVATAR_ROOT"] = PROJECT_ROOT
prepare_env["VENV_DIR"] = VENV_DIR
if ENABLE_SEARCH:
    prepare_env["ENABLE_SEARCH"] = "1"
    prepare_env["REF_LORA_DIR"] = REF_LORA_DIR
    prepare_env["STT_PKG_DIR"] = STT_PKG_DIR

run(
    f'bash "{PREPARE_SCRIPT}" --hf-token "{HF_TOKEN}"',
    cwd=PROJECT_ROOT, env=prepare_env, retries=1, timeout=PREPARE_TIMEOUT_SEC,
)
print("[ok] prepare_imtalker_personaplex.sh completed (includes the built-in --check-only preflight)")
if ENABLE_SEARCH:
    print("[ok] search dependencies/assets requested (peft, transformers override, isolated STT moshi, reference LoRA)")


## Step 3a — Web search API key (skipped if `ENABLE_SEARCH=False`)

Requested securely via `getpass`, same pattern as the Hugging Face token —
**never** hardcode this in a cell: notebooks get committed, shared, and
pasted into issues, and a key pasted here leaks with them. Only asked for if
both `ENABLE_SEARCH` and `WEB_SEARCH_ENABLED` are `True`. Get a free key from
your chosen provider (Tavily, Serper, or Bing) and set `WEB_SEARCH_PROVIDER`
in the Parameters cell to match.


In [ ]:
if ENABLE_SEARCH and WEB_SEARCH_ENABLED:
    import getpass as _getpass

    web_search_key = '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'
    if not web_search_key:
        raise RuntimeError("WEB_SEARCH_ENABLED is True but no web search API key was provided.")
    os.environ["WEB_SEARCH_API_KEY"] = web_search_key
    print("[ok] web search API key set for this process (never printed, never written to a cell).")
else:
    print("ENABLE_SEARCH or WEB_SEARCH_ENABLED is False -- skipping web-search key prompt.")


## Step 3b — Verify search assets (skipped if `ENABLE_SEARCH=False`)

`prepare_imtalker_personaplex.sh` (Step 3, run with `ENABLE_SEARCH=1`) already
fetched the reference LoRA (`REF_LORA_DIR`, downloaded from
`Darknsu/helium_lora_v1` with a hand-written `adapter_config.json`, since that
dataset repo doesn't publish one). That adapter teaches PersonaPlex to
correctly use the injected `<lookup>`/`<ref>` tags.

This cell confirms it is present before launch, and checks that
`IMTalker/search_helpers.py` and `IMTalker/conversation_logger.py` both exist
**and import cleanly inside the venv** — a missing/broken file here is the
most common cause of search silently degrading to "disabled" deep in the log
with no obvious error.


In [ ]:
if ENABLE_SEARCH:
    ref_lora_config = os.path.join(REF_LORA_DIR, "lora", "adapter_config.json")
    search_helpers_py = os.path.join(IMTALKER_DIR, "search_helpers.py")
    conv_logger_py = os.path.join(IMTALKER_DIR, "conversation_logger.py")
    latency_logger_py = os.path.join(IMTALKER_DIR, "latency_logger.py")
    missing_search = [
        p for p in (ref_lora_config, search_helpers_py, conv_logger_py, latency_logger_py)
        if not os.path.exists(p)
    ]
    if missing_search:
        raise RuntimeError(
            f"ENABLE_SEARCH is True but required assets/files are missing: {missing_search}. "
            f"If IMTalker/search_helpers.py, conversation_logger.py or latency_logger.py is missing, "
            f"the checked-out repo does not contain the routing/search code. Otherwise re-run Step 3 "
            f"with ENABLE_SEARCH set (prepare_imtalker_personaplex.sh), or set ENABLE_SEARCH = False."
        )
    print(f"[ok] reference LoRA present: {ref_lora_config}")
    print(f"[ok] IMTalker/search_helpers.py present")
    print(f"[ok] IMTalker/conversation_logger.py present")
    print(f"[ok] IMTalker/latency_logger.py present")

    # Importability check: catches search_helpers.py/conversation_logger.py
    # existing on disk but failing to import inside the actual venv (missing
    # dependency, syntax error, etc.) -- runs in the SAME venv the live server
    # itself uses.
    _import_probe = (
        "import sys, traceback\n"
        f"sys.path.insert(0, {IMTALKER_DIR!r})\n"
        "try:\n"
        "    import search_helpers, conversation_logger, latency_logger\n"
        "    print('IMPORT_OK')\n"
        "except Exception:\n"
        "    traceback.print_exc()\n"
        "    print('IMPORT_FAILED')\n"
    )
    import_check = subprocess.run([VENV_PYTHON, "-c", _import_probe], capture_output=True, text=True)
    print(import_check.stdout)
    print(import_check.stderr)
    if "IMPORT_OK" not in import_check.stdout:
        raise RuntimeError(
            "search_helpers.py / conversation_logger.py / latency_logger.py exist but failed to "
            "import in the venv (see traceback above) -- fix this before launching, or search will "
            "silently disable itself at server startup."
        )
    print("[ok] search_helpers / conversation_logger / latency_logger import cleanly in the venv")

    thinking_sound_path = os.path.join(PROJECT_ROOT, "bundled_assets", "ai-thinking-sound.wav")
    if os.path.exists(thinking_sound_path):
        print(f"[ok] thinking-sound WAV present: {thinking_sound_path}")
    else:
        print(
            f"[info] thinking-sound WAV not found at {thinking_sound_path} -- the avatar will stay "
            f"silent (not wrong, just less polished) while routing/searching instead of playing a "
            f"'thinking' cue. Optional: add a short looped clip there, or point "
            f"THINKING_SOUND_PATH (env var passed to run_imtalker_personaplex.sh) elsewhere."
        )
else:
    print("ENABLE_SEARCH is False - skipping search asset verification.")


## Step 3c — Component self-tests (skipped if `ENABLE_SEARCH=False`)

Each cell below loads exactly one component in isolation, in the same venv
the live server uses, and prints either `[ok] ...` lines or a full Python
traceback. Run these **before** launching (Step 7) to confirm each piece
actually works, rather than only discovering a silent failure deep in
`live_server.log` after the avatar is already running. These are read-only
sanity checks — they do not affect the live server process.

Order: **router** (the decision component — read this table to see the
normal-question-vs-search-question split you will actually get), **STT**,
**compressor**, **web search**, **latency logger**.


In [ ]:
if ENABLE_SEARCH:
    # Router self-test. Loads the Qwen model once, then prints the verdict for
    # a spread of questions that SHOULD need live data and questions that
    # should NOT. Read the table: it is the clearest preview of how the
    # assistant will behave. Tune ROUTER_THRESHOLD in the Parameters cell if
    # the split is not where you want it.
    _probe = f"""
import sys, traceback
sys.path.insert(0, {IMTALKER_DIR!r})
try:
    import search_helpers

    # 1) Rules-only pass -- no model, microseconds, no GPU needed.
    rule_cases = [
        ("Can you tell me your name?", False),
        ("How are you doing?", False),
        ("Who are you?", False),
        ("Can you hear me?", False),
        ("Good morning", False),
        ("thanks a lot", False),
        ("Hi, what is the weather today?", True),
        ("what is the gold price today", True),
        ("who won the match yesterday", True),
        ("how do I invest in Bitcoin", False),
        ("is it safe to invest in cryptocurrency?", False),
        ("how does the stock market work", False),
        ("when was Bitcoin created", False),
        ("what is Bitcoin?", None),
    ]
    print("--- Tier 0: layered instant rules ---")
    for q, expected in rule_cases:
        got, why = search_helpers.rule_route_explain(q)
        label = {{True: "SEARCH", False: "no search", None: "-> ask the model"}}[got]
        flag = "ok " if got == expected else "DIFF"
        print(f"  [{{flag}}] {{label:16s}} {{q!r}}")
        print(f"           {{why}}")

    # 2) Full router incl. the Qwen forward pass.
    cc = search_helpers.ContextCompressor(
        model_name={COMPRESSOR_MODEL!r}, device="cuda", quantize_4bit=True,
    )
    router = search_helpers.QueryRouter.from_compressor(
        cc, threshold=float({ROUTER_THRESHOLD!r}), use_rules=bool(int({ROUTER_RULES!r})),
    )
    print("")
    print("--- Tier 1: full router (rules + model) ---")
    cases = [
        "what is the gold price today",
        "what is the weather in Dhaka right now",
        "who is the current president of France",
        "explain how a transformer neural network works",
        "what is the capital of Japan",
        "tell me a joke",
        "is Tesla a good investment",
        "what is the population of Tokyo",
        "how I can invest in cryptocurrency?",
        "Can you tell me your name?",
        "How are you doing?",
    ]
    print(f"  {{'verdict':<12}} {{'via':<7}} {{'score':>6}}  {{'ms':>6}}  question")
    for q in cases:
        v = router.decide(q)
        verdict = "SEARCH" if v["needs_search"] else "no search"
        print(f"  {{verdict:<12}} {{v['source']:<7}} {{v['score']:>6.3f}}  "
              f"{{1000*v['elapsed_s']:>6.0f}}  {{q}}")
    print("")
    print("ROUTER_SELFTEST_PASSED")
except Exception:
    traceback.print_exc()
    print("ROUTER_SELFTEST_FAILED")
"""
    result = subprocess.run([VENV_PYTHON, "-c", _probe], capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    if "ROUTER_SELFTEST_PASSED" not in result.stdout:
        print("[FAIL] Router self-test failed -- see traceback above. Without the router, "
              "every turn is answered from the model's own knowledge and nothing is ever searched.")
else:
    print("ENABLE_SEARCH is False - skipping.")


In [ ]:
if ENABLE_SEARCH:
    _probe = f"""
import sys, traceback
sys.path.insert(0, {IMTALKER_DIR!r})
try:
    import search_helpers, torch
    moshi_stt = search_helpers.load_upstream_moshi_stt({STT_PKG_DIR!r})
    print("[ok] upstream moshi package loaded under alias moshi_stt (version:", getattr(moshi_stt, "__version__", "?"), ")")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    stt_info = moshi_stt.models.loaders.CheckpointInfo.from_hf_repo("kyutai/stt-1b-en_fr-candle")
    print("[ok] CheckpointInfo.from_hf_repo resolved")
    stt_mimi = stt_info.get_mimi(device=device)
    stt_lm = stt_info.get_moshi(device=device, dtype=torch.bfloat16)
    n_params = sum(p.numel() for p in stt_lm.parameters()) / 1e9
    print(f"[ok] STT model loaded: {{n_params:.2f}}B params on {{device}}")
    lm_gen = moshi_stt.models.LMGen(stt_lm, temp=0, temp_text=0.0)
    print("[ok] STT LMGen constructed:", hasattr(lm_gen, "step_with_extra_heads"))
    print("STT_SELFTEST_PASSED")
except Exception:
    traceback.print_exc()
    print("STT_SELFTEST_FAILED")
"""
    result = subprocess.run([VENV_PYTHON, "-c", _probe], capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    if "STT_SELFTEST_PASSED" not in result.stdout:
        print("[FAIL] STT self-test failed -- see traceback above. Turn detection and routing will not work without this.")
else:
    print("ENABLE_SEARCH is False - skipping.")


In [ ]:
if ENABLE_SEARCH:
    _probe = """
import sys, time, traceback
sys.path.insert(0, r'{imtalker}')
try:
    import search_helpers

    # 1) Extractive path -- pure Python, no GPU, should be ~0ms. This is what
    # COMPRESSOR_MODE=extractive_first tries BEFORE ever calling the LLM below.
    hits = [
        {{"id": "w1", "source": "warmup", "title": "Warmup",
          "text": "The current price of gold today is $4,068.60 per ounce, up 1.2% from yesterday.",
          "similarity_score": 0.9}},
    ]
    t0 = time.perf_counter()
    text, score = search_helpers.extract_best_sentence("what is the gold price today", hits)
    extract_ms = 1000.0 * (time.perf_counter() - t0)
    print(f"[ok] extractive: {{extract_ms:.1f}}ms score={{score:.2f}} text={{text!r}}")

    # 2) LLM compressor path -- measures the ACTUAL latency on this GPU, so
    # the fix (extractive_first skipping this call in the common case) can be
    # verified against a real number instead of the 2026-09-05 log's 2-5s.
    cc = search_helpers.ContextCompressor(model_name="{model}", device="{device}", quantize_4bit=True)
    t0 = time.perf_counter()
    result = cc.compress(
        question="What is the deferred tax liability?",
        chunks=[{{"id": "warmup", "source": "warmup", "text": "Deferred tax liability is 1.01.", "similarity_score": 1.0}}],
    )
    llm_ms = 1000.0 * (time.perf_counter() - t0)
    print(f"[ok] Qwen compressor loaded and responded in {{llm_ms:.0f}}ms: {{result!r}}")
    if llm_ms > 1500:
        print(f"[warn] LLM compression took {{llm_ms:.0f}}ms -- this is the latency COMPRESSOR_MODE="
              f"extractive_first is designed to skip in the common case (see run_imtalker_personaplex.sh). "
              f"If most real questions are NOT being answered by extraction, check "
              f"EXTRACTIVE_CONFIDENCE_THRESHOLD and system_runtime.log's 'Compressor extractive_not_confident' lines.")
    print("COMPRESSOR_SELFTEST_PASSED")
except Exception:
    traceback.print_exc()
    print("COMPRESSOR_SELFTEST_FAILED")
""".format(imtalker=IMTALKER_DIR, model=COMPRESSOR_MODEL, device=COMPRESSOR_DEVICE)
    result = subprocess.run([VENV_PYTHON, "-c", _probe], capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    if "COMPRESSOR_SELFTEST_PASSED" not in result.stdout:
        print("[FAIL] Compressor self-test failed -- see traceback above. The router shares this "
              "model, so BOTH routing and search are disabled if it cannot load: every turn would "
              "then be answered from the model's own knowledge.")
else:
    print("ENABLE_SEARCH is False - skipping.")

In [ ]:
if ENABLE_SEARCH and WEB_SEARCH_ENABLED:
    _probe = f"""
import sys, traceback, os
sys.path.insert(0, {IMTALKER_DIR!r})
try:
    import search_helpers
    key = os.environ.get("WEB_SEARCH_API_KEY", "")
    if not key:
        print("[skip] WEB_SEARCH_API_KEY not set in this process's environment")
    else:
        hits = search_helpers.web_search_query_sync(
            "current weather in New York", key, {WEB_SEARCH_PROVIDER!r}, 3, 5.0
        )
        print(f"[ok] web search returned {{len(hits)}} hits")
        for h in hits:
            print(f"    score={{h['similarity_score']:.3f}} source={{h['source']}} text={{h['text'][:80]!r}}")
    print("WEB_SEARCH_SELFTEST_PASSED")
except Exception:
    traceback.print_exc()
    print("WEB_SEARCH_SELFTEST_FAILED")
"""
    result = subprocess.run([VENV_PYTHON, "-c", _probe], capture_output=True, text=True, env=os.environ.copy())
    print(result.stdout)
    print(result.stderr)
    if "WEB_SEARCH_SELFTEST_PASSED" not in result.stdout:
        print("[FAIL] Web search self-test failed -- see traceback/output above.")
else:
    print("ENABLE_SEARCH or WEB_SEARCH_ENABLED is False - skipping.")


In [ ]:
# Latency-log self-test: writes a fake turn into a scratch directory using the
# real LatencyLogger, so a broken/missing latency_logger.py is caught here
# rather than by an empty latency_*.log after a live conversation.
if ENABLE_SEARCH:
    _probe = f"""
import sys, tempfile, os, time, traceback
sys.path.insert(0, {IMTALKER_DIR!r})
try:
    from conversation_logger import ConversationLogger
    d = tempfile.mkdtemp(prefix="latency_selftest_")
    cl = ConversationLogger(log_dir=d, session_id="selftest")
    t0 = time.perf_counter()
    cl.latency.start_turn(1, t0=t0, transcript="what is the gold price today")
    cl.latency.stage(1, "router_model", 0.08, note="model: score 0.930")
    cl.latency.mark(1, "decision_made", "search online")
    cl.latency.stage(1, "web_search", 1.42, note="tavily: 5 result(s)")
    cl.latency.stage(1, "compression", 0.63, note="18 token(s) out")
    cl.latency.count(1, compressor_output_tokens=18, compressor_generated_tokens=21)
    cl.latency.mark(1, "first_word")
    cl.latency.mark(1, "answer_complete", at=t0 + 4.0)
    cl.latency.finish_turn(1, response="Gold is about 4,068 dollars an ounce.")
    written = sorted(f for f in os.listdir(d) if f.startswith("latency_"))
    body = open(os.path.join(d, "latency_selftest.log"), encoding="utf-8").read()
    assert written == ["latency_selftest.jsonl", "latency_selftest.log"], written
    assert "compressor_output_tokens" in body, "compressor token counts missing"
    assert "question -> FIRST spoken word" in body, "headline latencies missing"
    print(f"[ok] latency logger wrote {{written}} ({{len(body)}} bytes) in {{d}}")
    print("LATENCY_SELFTEST_PASSED")
except Exception:
    traceback.print_exc()
    print("LATENCY_SELFTEST_FAILED")
"""
    result = subprocess.run([VENV_PYTHON, "-c", _probe], capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    if "LATENCY_SELFTEST_PASSED" not in result.stdout:
        print("[FAIL] Latency log self-test failed -- see traceback above. The live server still "
              "runs and still logs conversations; only latency_<session>.log is lost.")
else:
    print("ENABLE_SEARCH is False - skipping.")


## Step 4 — Post-install GPU / CUDA confirmation

Independent double-check (beyond the script's own preflight) that the venv's
torch matches the required pin and CUDA is available, using the same probe
the old notebook used.


In [ ]:
torch_info = get_torch_info()
print(f"[info] torch={torch_info['version']} cuda_available={torch_info['cuda_available']} "
      f"cuda_version={torch_info['cuda_version']} device={torch_info['device_name']}")

if torch_info["version"] != EXPECTED_TORCH_VERSION:
    raise RuntimeError(f"torch pin violated: {torch_info['version']} != {EXPECTED_TORCH_VERSION}")
if not torch_info["cuda_available"]:
    raise RuntimeError("torch.cuda.is_available() is False inside the venv after installation.")
if torch_info["device_name"] and "5090" in torch_info["device_name"]:
    print(f"[ok] GPU confirmed: {torch_info['device_name']}")
else:
    print(f"[warn] GPU device name '{torch_info['device_name']}' does not mention 5090 — continuing anyway")


## Step 5 — Pre-launch port check

`run_imtalker_personaplex.sh` refuses to start (rather than recovering) if
the port is already occupied. This cell mirrors the old notebook's recovery
behaviour: if the port is held by a *previous run of this same notebook*
(tracked via `PID_PATH`), it is terminated automatically; if held by anything
else, the cell fails fast with instructions rather than guessing.


In [ ]:
def check_port_and_recover(port):
    if not port_listening("127.0.0.1", port):
        print(f"[ok] port {port} is free")
        return
    pids_on_port = find_pids_on_port(port)
    prior_pid = None
    if os.path.exists(PID_PATH):
        prior_pid = open(PID_PATH).read().strip()
    if prior_pid and prior_pid in pids_on_port:
        print(f"[recovery] port {port} held by a previous notebook-launched server (pid {prior_pid}); terminating it")
        subprocess.run(["kill", "-9", prior_pid])
        time.sleep(2)
        if port_listening("127.0.0.1", port):
            raise RuntimeError(f"Port {port} still in use after terminating prior pid {prior_pid}")
        print(f"[ok] port {port} freed")
    else:
        raise RuntimeError(
            f"Port {port} is already in use by pid(s) {pids_on_port}, which were not started by this "
            f"notebook. Stop that process manually (or choose a different PORT) before launching."
        )


check_port_and_recover(PORT)


## Step 6 — Inspect `run_imtalker_personaplex.sh` and resolve runtime overrides

The notebook never reimplements `run_imtalker_personaplex.sh`'s argument
list — it reads and prints the actual script, then prepares only the
environment-variable overrides the script already supports (`PORT`,
`CUDA_VISIBLE_DEVICES`, `VOICE_PROMPT`, `TEXT_PROMPT` / `TEXT_PROMPT_FILE`,
`PROMPT_CACHE`, plus `SPEECH2AVATAR_ROOT` / `VENV_DIR`).


In [ ]:
with open(RUN_SCRIPT) as f:
    run_script_contents = f.read()
print(run_script_contents)


In [ ]:
env_overrides = {
    "SUPPRESS_MEDIA_WATCHDOG_SEC": SUPPRESS_MEDIA_WATCHDOG_SEC,
    "MAX_EVENT_BACKLOG_SEC": MAX_EVENT_BACKLOG_SEC,
    # The standing-latency valve. Read by run_imtalker_personaplex.sh and
    # passed to the server as --max_input_buffer_sec; without it the
    # microphone backlog is unbounded and every turn inherits it forever.
    "MAX_INPUT_BUFFER_SEC": MAX_INPUT_BUFFER_SEC,
    "SPEECH2AVATAR_ROOT": PROJECT_ROOT,
    "VENV_DIR": VENV_DIR,
    "PORT": str(PORT),
    "CUDA_VISIBLE_DEVICES": str(CUDA_VISIBLE_DEVICES),
    "LOGS_DIR": LOGS_DIR,
}
if VOICE_PROMPT:
    env_overrides["VOICE_PROMPT"] = VOICE_PROMPT
if TEXT_PROMPT:
    env_overrides["TEXT_PROMPT"] = TEXT_PROMPT
if TEXT_PROMPT_FILE:
    env_overrides["TEXT_PROMPT_FILE"] = TEXT_PROMPT_FILE
if PROMPT_CACHE:
    env_overrides["PROMPT_CACHE"] = PROMPT_CACHE

if ENABLE_SEARCH:
    env_overrides["ENABLE_SEARCH"] = "1"
    env_overrides["REF_LORA_DIR"] = REF_LORA_DIR
    env_overrides["STT_PKG_DIR"] = STT_PKG_DIR
    env_overrides["CONVERSATION_LOG_DIR"] = CONVERSATION_LOG_DIR
    env_overrides["ROUTER_THRESHOLD"] = str(ROUTER_THRESHOLD)
    env_overrides["ROUTER_RULES"] = str(ROUTER_RULES)
    env_overrides["COMPRESSOR_MODEL"] = COMPRESSOR_MODEL
    env_overrides["COMPRESSOR_DEVICE"] = COMPRESSOR_DEVICE
    env_overrides["COMPRESSOR_MODE"] = COMPRESSOR_MODE
    env_overrides["EXTRACTIVE_CONFIDENCE_THRESHOLD"] = str(EXTRACTIVE_CONFIDENCE_THRESHOLD)
    env_overrides["MAX_SUPPRESS_SEC"] = str(MAX_SUPPRESS_SEC)
    env_overrides["INJECT_TOKENS_PER_TICK"] = str(INJECT_TOKENS_PER_TICK)
    env_overrides["POST_INJECT_WATCHDOG_SEC"] = str(POST_INJECT_WATCHDOG_SEC)
    env_overrides["INJECT_TOKENS_PER_TICK"] = str(INJECT_TOKENS_PER_TICK)
    env_overrides["REF_AUDIO_DRAIN_SEC"] = str(REF_AUDIO_DRAIN_SEC)
    if WEB_SEARCH_ENABLED:
        env_overrides["WEB_SEARCH_ENABLED"] = "1"
        env_overrides["WEB_SEARCH_PROVIDER"] = WEB_SEARCH_PROVIDER
        env_overrides["WEB_SEARCH_API_KEY"] = os.environ.get("WEB_SEARCH_API_KEY", "")
    else:
        # Explicit 0 so run_imtalker_personaplex.sh does not auto-enable web
        # search just because a key happens to be exported in this environment.
        env_overrides["WEB_SEARCH_ENABLED"] = "0"

print("Resolved environment overrides for run_imtalker_personaplex.sh:")
for k, v in env_overrides.items():
    print(f"  {k}={'***' if k == 'WEB_SEARCH_API_KEY' else v}")

## Step 7 — Launch the live server

`run_imtalker_personaplex.sh` runs its preflight again and then `exec`s the
server in the foreground, so this notebook launches it as a background
process (like the old notebook did with `run_live.sh`), logs to
`live_server.log`, and records the PID for the stop/recovery cells below.
Because the script uses `exec`, the recorded PID stays valid for the actual
Python server process (Unix `exec` replaces the process image but keeps the
PID).


In [ ]:
launch_env = os.environ.copy()
launch_env.update(env_overrides)

log_file = open(LOG_PATH, "w")
live_proc = subprocess.Popen(
    ["bash", RUN_SCRIPT], cwd=PROJECT_ROOT, env=launch_env,
    stdout=log_file, stderr=subprocess.STDOUT,
)
with open(PID_PATH, "w") as f:
    f.write(str(live_proc.pid))

print(f"[ok] launched live server: pid={live_proc.pid}")
print(f"     log file: {LOG_PATH}")
print(f"     pid file: {PID_PATH}")


## Step 8 — Wait for healthy startup

Polls the log and the listening port simultaneously. Healthy markers are
drawn from the script's own preflight banner (`Preflight OK: try_vad2, ...`)
and the standard Uvicorn startup banner. On timeout or early process exit, it
dumps the log tail and common-failure hints instead of hanging forever.


In [ ]:
HEALTHY_MARKERS = {
    "preflight OK": "Preflight OK: try_vad2",
    "Uvicorn running on host:port": f"Uvicorn running on http://{HOST}:{PORT}",
}

ERROR_HINTS = {
    "CUDA out of memory": "GPU out of memory — reduce concurrent load or confirm the pod truly has an RTX 5090 with enough VRAM",
    "Traceback (most recent call last)": "a Python exception occurred during startup — see the traceback above in the log",
    "Missing required file": "a referenced checkpoint/asset path is missing — re-run Step 3 (prepare script)",
    "Checksum mismatch": "a protected runtime file failed SHA-256 verification — re-run Step 3 to re-download it",
    "CUDA error": "a CUDA/driver mismatch occurred — re-check Step 1 (nvidia-smi) and the torch CUDA build",
    "Port": "the port was taken by another process after the Step 5 check — re-run Step 5",
}


def wait_for_healthy(timeout=STARTUP_TIMEOUT_SEC, poll_interval=POLL_INTERVAL_SEC):
    start = time.time()
    last_print = 0.0
    while time.time() - start < timeout:
        if live_proc.poll() is not None:
            tail_log(150)
            raise RuntimeError(
                f"live server process exited early with code {live_proc.returncode}; see log tail above"
            )

        log_text = ""
        if os.path.exists(LOG_PATH):
            with open(LOG_PATH, "r", errors="ignore") as f:
                log_text = f.read()

        markers_ok = {label: (needle in log_text) for label, needle in HEALTHY_MARKERS.items()}
        port_ok = port_listening("127.0.0.1", PORT)

        if all(markers_ok.values()) and port_ok:
            print("[ok] live server is healthy")
            for label, ok in markers_ok.items():
                print(f"  [OK] {label}")
            print(f"  [OK] port {PORT} listening")
            return True

        if time.time() - last_print > 15:
            elapsed = int(time.time() - start)
            print(f"[..] waiting ({elapsed}s/{timeout}s) markers={markers_ok} port_listening={port_ok}")
            last_print = time.time()

        time.sleep(poll_interval)

    print("[error] timed out waiting for healthy startup; log tail:")
    tail_log(150)
    log_text = ""
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH, "r", errors="ignore") as f:
            log_text = f.read()
    for needle, hint in ERROR_HINTS.items():
        if needle in log_text:
            print(f"[diagnosis] found '{needle}' in log -> {hint}")
    raise TimeoutError("Live server did not become healthy within the timeout; see diagnostics above")


wait_for_healthy()


## Step 9 — HTTP confirmation

Final confirmation that the FastAPI/Uvicorn app is actually serving HTTP on
`0.0.0.0:8998`, plus the two health-check URLs documented in `live_8998.md`
(`/` and `/assets/robert_idle_10s.mp4`), not just that the port is open.


In [ ]:
import urllib.error
import urllib.request

for path in ["/", "/assets/robert_idle_10s.mp4"]:
    url = f"http://127.0.0.1:{PORT}{path}"
    try:
        with urllib.request.urlopen(url, timeout=10) as resp:
            print(f"[ok] HTTP {resp.status} from {url}")
    except urllib.error.URLError as e:
        tail_log(80)
        raise RuntimeError(f"HTTP check failed against {url}: {e}")

with open(LOG_PATH, "r", errors="ignore") as f:
    final_log = f.read()
assert f"Uvicorn running on http://{HOST}:{PORT}" in final_log, "Uvicorn startup banner not found in log"

print()
print("=" * 72)
print("SUCCESS: PersonaPlex + IMTalker (updated / try_vad2) live server is running")
print(f"  Internal: http://{HOST}:{PORT}")
print(f"  Open port {PORT} through your RunPod pod's proxy/port mapping to access")
print(f"  the browser UI (index_v3_binary_fullscreen_robot_try_vad2.html is served at /).")
print(f"  PID: {open(PID_PATH).read().strip()}   Log: {LOG_PATH}")
print("=" * 72)


## Step 10 — Search pipeline status (skipped if `ENABLE_SEARCH=False`)

Informational only, never gates the health check above. Loading failures are
logged and degrade gracefully (the avatar still boots and still talks, just
without routing/search); this surfaces whether that happened AND prints the
actual failure line(s) with surrounding context wherever they are in the log
— not just the tail, which for a long-running server almost certainly no
longer contains the startup-time failure by the time you look.


In [ ]:
if ENABLE_SEARCH:
    with open(LOG_PATH, "r", errors="ignore") as f:
        _log_lines = f.readlines()
    _log_text = "".join(_log_lines)
    _markers = {
        "reference LoRA loaded": "reference LoRA loaded" in _log_text,
        "STT model loaded": "STT model loaded" in _log_text,
        "context compressor ready": "[compressor] ready" in _log_text,
        "query router ready": "query router ready" in _log_text,
    }
    print("Search pipeline status:")
    for label, ok in _markers.items():
        print(f"  [{'OK' if ok else '??'}] {label}")

    if not _markers["query router ready"]:
        print("\n[warn] the query router never came up -- every turn will be answered from the")
        print("       model's own knowledge and nothing will ever be searched. The router shares")
        print("       the compressor's model, so check the compressor lines first.")

    _tok = [l for l in _log_lines if "stt tokenizer=" in l]
    if _tok:
        print("\nTokenizers:")
        for l in _tok[-1:]:
            print("  " + l.strip())
    _tokwarn = [l for l in _log_lines if "tokenizers look identical" in l]
    if _tokwarn:
        print("  [!!] " + _tokwarn[-1].strip())

    _rejected = [l for l in _log_lines if "| REJECT" in l]
    if _rejected:
        print(f"\n[warn] {len(_rejected)} transcript(s) were rejected as unusable "
              f"(wrong script for an en/fr STT model). Most recent:")
        for l in _rejected[-3:]:
            print("  " + l.rstrip())

    _needles = ("search disabled", "compressor disabled", "router disabled",
                "[conversation.", "kind=error")
    _hit_line_nums = [i for i, line in enumerate(_log_lines) if any(n in line for n in _needles)]
    if _hit_line_nums:
        print(f"\n[warn] found {len(_hit_line_nums)} error/disable line(s) in the log "
              f"(showing each with 5 lines of context):")
        _shown = set()
        for ln in _hit_line_nums:
            start, end = max(0, ln - 2), min(len(_log_lines), ln + 6)
            if start in _shown:
                continue
            _shown.add(start)
            print(f"  --- log lines {start}-{end} ---")
            for l in _log_lines[start:end]:
                print("  " + l.rstrip())
    else:
        print("\n[ok] no search-related disable/error lines found anywhere in the log.")
else:
    print("ENABLE_SEARCH is False -- routing/search not requested for this launch.")


## Step 11 — Verify system_runtime.log and conversation.log

Confirms both always-on log files (independent of `ENABLE_SEARCH`) exist,
are non-empty, and every line carries a millisecond-precision timestamp
(`YYYY-MM-DD HH:MM:SS.mmm`). `system_runtime.log` should already show the
model/adapter loads and GPU info from Steps 7-9 above; `conversation.log`
should show at least the startup `component_status` line (real conversation
turns only appear there once `ENABLE_SEARCH=True` and someone has spoken to
the avatar).

In [ ]:
import re

_TS_RE = re.compile(r"^\[\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{3}\]")


def _check_log_file(path, label, min_lines=1):
    if not os.path.exists(path):
        print(f"[FAIL] {label} does not exist: {path}")
        return False
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = [ln for ln in f.read().splitlines() if ln.strip()]
    if len(lines) < min_lines:
        print(f"[FAIL] {label} exists but has no content yet: {path}")
        return False
    timestamped = [ln for ln in lines if _TS_RE.match(ln)]
    ratio = len(timestamped) / len(lines)
    print(f"[info] {label}: {path}")
    print(f"       {len(lines)} line(s), {len(timestamped)} with a millisecond timestamp ({ratio:.0%})")
    print(f"       last line: {lines[-1][:160]}")
    if ratio < 0.5:
        print(f"[FAIL] {label}: fewer than half the lines carry the expected "
              f"'[YYYY-MM-DD HH:MM:SS.mmm]' millisecond timestamp")
        return False
    print(f"[ok] {label} contains millisecond-timestamped lines")
    return True


system_ok = _check_log_file(SYSTEM_LOG_PATH, "system_runtime.log")
conversation_ok = _check_log_file(CONVERSATION_FLOW_LOG_PATH, "conversation.log")

if system_ok and conversation_ok:
    print("\nLOGGING_SELFTEST_PASSED")
else:
    print("\n[FAIL] LOGGING_SELFTEST_FAILED -- see the [FAIL] lines above.")
    if not ENABLE_SEARCH:
        print("       Note: with ENABLE_SEARCH=False, conversation.log will only ever contain the")
        print("       startup component_status line -- that is expected, not a bug. It still needs")
        print("       to exist with a valid millisecond timestamp on that line.")

## Operational cells (run any time)

These are safe to re-run independently after the server is up.


In [ ]:
# Tail the live server log
tail_log(200)


In [ ]:
# Full diagnostics — safe to run any time, including during a failure
def run_full_diagnostics():
    print("== GPU ==")
    run("nvidia-smi", check=False)

    print("\n== Torch / CUDA (venv) ==")
    try:
        print(get_torch_info())
    except Exception as e:
        print(f"[error] torch check failed: {e}")

    print(f"\n== Port {PORT} ==")
    print(f"listening: {port_listening('127.0.0.1', PORT)}  pids: {find_pids_on_port(PORT)}")

    print("\n== Preflight (--check-only) ==")
    check_env = os.environ.copy()
    check_env["SPEECH2AVATAR_ROOT"] = PROJECT_ROOT
    check_env["VENV_DIR"] = VENV_DIR
    run(f'bash "{RUN_SCRIPT}" --check-only', cwd=PROJECT_ROOT, env=check_env, check=False)

    print("\n== pip check (venv) ==")
    run(f"{VENV_ACTIVATE} && pip check", check=False)

    print("\n== Log tail ==")
    tail_log(100)


run_full_diagnostics()


In [ ]:
# Per-turn conversation trace. Reads the JSONL written by
# IMTalker/conversation_logger.py and replays each turn in order:
#   HEARD -> DECIDE (search or not, and why) -> SEARCH -> GROUND -> DONE -> REPLIED
# Re-run any time while chatting. Requires ENABLE_SEARCH=True and a
# CONVERSATION_LOG_DIR.
import glob
import json as _json
from collections import Counter, defaultdict

if ENABLE_SEARCH and CONVERSATION_LOG_DIR:
    jsonl_files = sorted(glob.glob(os.path.join(CONVERSATION_LOG_DIR, "conversation_*.jsonl")))
    if not jsonl_files:
        print(f"[info] no conversation_*.jsonl files yet under {CONVERSATION_LOG_DIR} "
              f"(nothing has happened in a conversation yet, or the server hasn't started).")
    else:
        latest = jsonl_files[-1]
        print(f"Reading {latest}\n")
        events = []
        with open(latest, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    events.append(_json.loads(line))
                except _json.JSONDecodeError:
                    continue

        counts = Counter(e.get("kind", "?") for e in events)
        decisions = [e for e in events if e.get("kind") == "router_decision"]
        rejected = [e for e in events if e.get("kind") == "transcript_rejected"]
        n_search = sum(1 for e in decisions if e.get("needs_search"))
        print(f"Total events : {len(events)}")
        print(f"Turns routed : {len(decisions)}  "
              f"({n_search} searched online, {len(decisions) - n_search} answered directly)")
        if decisions:
            by_src = Counter(e.get("source", "?") for e in decisions)
            print(f"Decided by   : " + ", ".join(f"{k}={v}" for k, v in by_src.most_common()))
        if rejected:
            print(f"[warn] {len(rejected)} transcript(s) rejected as unusable script")

        turns = defaultdict(list)
        for e in events:
            t = e.get("turn", e.get("turn_epoch"))
            if t is not None:
                turns[t].append(e)

        LABEL = {
            "user_transcript": "HEARD",
            "transcript_rejected": "REJECT",
            "router_decision": "DECIDE",
            "turn_search": "SEARCH",
            "turn_ground": "GROUND",
            "turn_done": "DONE",
            "assistant_reply": "REPLIED",
        }
        print("\n--- last 5 turns ---")
        for t in sorted(turns)[-5:]:
            print(f"\nTURN {t}")
            for e in turns[t]:
                k = e.get("kind")
                lab = LABEL.get(k)
                if not lab:
                    continue
                if k == "user_transcript":
                    body = f'"{e.get("transcript", "")}"'
                elif k == "transcript_rejected":
                    st = e.get("script_stats") or {}
                    body = (f'DISCARDED ({st.get("non_latin_ratio", 0):.0%} non-Latin) '
                            f'"{str(e.get("transcript", ""))[:60]}"  ids={(e.get("token_ids") or [])[:12]}')
                elif k == "router_decision":
                    body = (f'{"SEARCH ONLINE" if e.get("needs_search") else "ANSWER DIRECTLY":<15} '
                            f'via={e.get("source", "?"):<6} score={e.get("score", 0):.3f}\n'
                            f'{"":<12}why: {e.get("reason", "")}')
                elif k == "turn_search":
                    body = (f'{e.get("provider", "?")}: {e.get("n_found", 0)} found, '
                            f'{e.get("n_kept", 0)} kept ({e.get("elapsed_s", 0):.2f}s)')
                elif k == "turn_ground":
                    body = f'"{e.get("grounding", "")}"' + (" (extractive)" if e.get("used_fallback") else "")
                elif k == "turn_done":
                    body = f'{e.get("outcome", "")} ({e.get("total_s", 0):.2f}s)'
                else:
                    body = f'"{str(e.get("response", ""))[:110]}"'
                print(f"  {lab:<8} {body}")

        errors = [e for e in events if e.get("kind") == "error"]
        if errors:
            print(f"\n--- {len(errors)} error event(s) -- most recent traceback ---")
            print(errors[-1].get("traceback", "")[-3000:])
else:
    print("ENABLE_SEARCH is False, or CONVERSATION_LOG_DIR is empty -- nothing to show.")


In [ ]:
# Per-turn latency report. Reads the JSONL written by IMTalker/latency_logger.py.
# Re-run any time while chatting.
import glob
import json as _json
from statistics import mean

SHOW_LATENCY_LOG_TAIL = False   # True -> also print the raw human-readable blocks
LATENCY_TURNS_SHOWN = 5

STAGE_ORDER = [
    ("stt_decode", "speech-to-text decode"),
    ("transcript_check", "transcript sanity check"),
    ("rule_route", "quick rule check (search?)"),
    ("router_model", "router model (search?)"),
    ("web_search", "online search"),
    ("search_filter", "filter/sort results"),
    ("compression", "compress -> one sentence (LLM)"),
    ("compression_extractive", "extract best sentence (no LLM)"),
    ("compression_llm", "compress -> one sentence (LLM)"),
    ("compression_fallback", "extractive fallback summary"),
    ("ref_encode", "tokenize + trim grounding"),
    ("lookup_inject", "inject <lookup> wait note"),
    ("ref_inject", "inject <ref> grounding"),
    ("gen_lm", "answer generation (Moshi LM)"),
    ("gen_audio_decode", "answer generation (audio decode)"),
    ("gen_audio_encode", "answer generation (audio encode)"),
    ("thinking_sound", "thinking sound played (overlaps)"),
]

if ENABLE_SEARCH and CONVERSATION_LOG_DIR:
    files = sorted(glob.glob(os.path.join(CONVERSATION_LOG_DIR, "latency_*.jsonl")))
    if not files:
        print(f"[info] no latency_*.jsonl files yet under {CONVERSATION_LOG_DIR}. A turn is "
              f"written only once it is complete -- ask a question, then ask another one "
              f"(or stop the session), and re-run this cell.")
    else:
        latest = files[-1]
        print(f"Reading {latest}\n")
        turns = []
        with open(latest, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    turns.append(_json.loads(line))
                except _json.JSONDecodeError:
                    continue

        if not turns:
            print("[info] the file exists but no turn has been completed yet.")
        else:
            def avg(field):
                vals = [float(t[field]) for t in turns if t.get(field) is not None]
                return mean(vals) if vals else None

            searched = [t for t in turns if t.get("stage_web_search_s")]
            print(f"Turns completed        : {len(turns)}  ({len(searched)} used an online search)")
            # NOTE: question_to_first_word_s is when the MODEL produced audio,
            # not when the user heard it. Those are very different numbers in
            # this pipeline, and reporting only the first one is why a session
            # could look like "0.06s" here while a stopwatch said 7-8s. The
            # delivery fields below are the ones that match what a person
            # actually experiences -- read question_to_audio_out_s first.
            for field, label in (
                ("question_to_first_word_s", "question -> first word GENERATED"),
                ("question_to_audio_out_s", "question -> first audio HEARD (real latency)"),
                ("generated_to_sent_s", "  of which: pipeline delivery delay"),
                ("produce_to_publish_s", "    ...chunk fill + FM + render"),
                ("audio_queue_wait_s", "    ...waiting in the send queue"),
                ("question_to_answer_complete_s", "question -> answer fully spoken"),
                ("question_to_decision_s", "question -> search decision"),
                ("question_to_ref_injected_s", "question -> grounding injected"),
            ):
                v = avg(field)
                if v is not None:
                    worst = max(float(t[field]) for t in turns if t.get(field) is not None)
                    print(f"avg {label:<32} {v:6.2f}s   (worst {worst:.2f}s)")

            print("\n--- average component cost across all turns (only turns where it ran) ---")
            for key, label in STAGE_ORDER:
                field = f"stage_{key}_s"
                vals = [float(t[field]) for t in turns if t.get(field)]
                if vals:
                    print(f"  {label:<34} {mean(vals):6.3f}s   n={len(vals):<3} max={max(vals):.3f}s")

            comp = [t for t in turns if t.get("compressor_output_tokens") is not None]
            if comp:
                print("\n--- compressor output size ---")
                out = [int(t["compressor_output_tokens"]) for t in comp]
                gen = [int(t.get("compressor_generated_tokens", 0)) for t in comp]
                capped = sum(1 for t in comp if t.get("compressor_hit_token_cap"))
                print(f"  calls                        {len(comp)}")
                print(f"  tokens OUT (cleaned)         avg {mean(out):.1f}  min {min(out)}  max {max(out)}")
                print(f"  tokens generated (raw)       avg {mean(gen):.1f}  max {max(gen)}")
                if capped:
                    print(f"  [warn] {capped} call(s) hit max_new_tokens -- the sentence was cut off, "
                          f"raise --compressor_max_new_tokens")

            print(f"\n--- last {LATENCY_TURNS_SHOWN} turns ---")
            for t in turns[-LATENCY_TURNS_SHOWN:]:
                first = t.get("question_to_first_word_s")
                heard = t.get("question_to_audio_out_s")
                done = t.get("question_to_answer_complete_s")
                print(f"\nTURN {t.get('turn')}  \"{str(t.get('transcript',''))[:70]}\"")
                print(f"  first word {'n/a' if first is None else f'{first:.2f}s'} | "
                      f"answer done {'n/a' if done is None else f'{done:.2f}s'} | "
                      f"{t.get('outcome') or ('searched' if t.get('stage_web_search_s') else 'answered directly')}")
                # The number a stopwatch measures, and where it went.
                if heard is not None:
                    parts = [f"HEARD at {heard:.2f}s"]
                    if t.get("generated_to_sent_s") is not None:
                        parts.append(f"delivery {float(t['generated_to_sent_s']):.2f}s")
                    if t.get("produce_to_publish_s") is not None:
                        parts.append(f"produce {float(t['produce_to_publish_s']):.2f}s")
                    if t.get("audio_queue_wait_s") is not None:
                        parts.append(f"queue {float(t['audio_queue_wait_s']):.2f}s")
                    if t.get("audio_q_depth_at_stream_start") is not None:
                        parts.append(f"audio_q={t['audio_q_depth_at_stream_start']}")
                    print("  " + " | ".join(parts))
                if t.get("input_backlog_s_at_first_word") is not None:
                    backlog = float(t["input_backlog_s_at_first_word"])
                    flag = "  <-- at the cap, GPU behind real time" if backlog >= 1.9 else ""
                    print(f"  mic backlog at first word {backlog:.2f}s{flag}")
                biggest = sorted(
                    ((k[len('stage_'):-len('_s')], v) for k, v in t.items()
                     if k.startswith('stage_') and k.endswith('_s') and isinstance(v, (int, float))),
                    key=lambda kv: kv[1], reverse=True,
                )[:4]
                if biggest:
                    print("  slowest: " + ", ".join(f"{n}={v:.2f}s" for n, v in biggest))
                if t.get("compressor_output_tokens") is not None:
                    line = (f"  compressor: {t['compressor_output_tokens']} token(s) out "
                            f"of {t.get('compressor_generated_tokens')} generated")
                    if t.get("grounding_tokens_injected") is not None:
                        line += f", injected into the model as {t['grounding_tokens_injected']} token(s)"
                    print(line)

        if SHOW_LATENCY_LOG_TAIL:
            text_path = latest[:-len(".jsonl")] + ".log"
            if os.path.exists(text_path):
                print("\n" + "=" * 80)
                print(f"raw tail of {text_path}\n")
                with open(text_path, "r", encoding="utf-8", errors="ignore") as f:
                    print("".join(f.readlines()[-160:]))
else:
    print("ENABLE_SEARCH is False, or CONVERSATION_LOG_DIR is empty -- nothing to show.")

In [ ]:
# Stop switch — set STOP_SERVER = True and re-run this cell to terminate the live server.
STOP_SERVER = False


def stop_server():
    if not os.path.exists(PID_PATH):
        print("[info] no pid file found; nothing to stop")
        return
    pid = open(PID_PATH).read().strip()
    print(f"[stop] terminating live server pid {pid}")
    subprocess.run(["pkill", "-9", "-P", pid], check=False)
    subprocess.run(["kill", "-9", pid], check=False)
    os.remove(PID_PATH)
    print("[ok] server stopped")


if STOP_SERVER:
    stop_server()
else:
    print("STOP_SERVER is False — set it to True above and re-run this cell to stop the server.")
